In [1]:
#  What This Code Does:
# - Simulates realistic customer behavior data.
# - Uses a logistic function to model purchase likelihood based on engagement.
# - Creates a labeled dataset for binary classification (purchased = 0 or 1).
# - Prints a preview and the overall purchase rate




import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Create sample customer data
np.random.seed(42)
n_customers = 1000

# Generate realistic business data
time_on_site = np.random.normal(10, 5, n_customers)  # minutes
pages_viewed = np.random.poisson(5, n_customers)     # number of pages
cart_value = np.random.exponential(50, n_customers)  # dollars
previous_purchases = np.random.binomial(5, 0.3, n_customers)  # count

# Create the outcome: probability increases with engagement
linear_combination = (
    0.1 * time_on_site + 
    0.2 * pages_viewed + 
    0.01 * cart_value + 
    0.5 * previous_purchases - 3
)
probabilities = 1 / (1 + np.exp(-linear_combination))
purchased = np.random.binomial(1, probabilities, n_customers)

# Create DataFrame
data = pd.DataFrame({
    'time_on_site': time_on_site,
    'pages_viewed': pages_viewed,
    'cart_value': cart_value,
    'previous_purchases': previous_purchases,
    'purchased': purchased
})

print("Sample of our customer data:")
print(data.head())
print(f"\nPurchase rate: {data['purchased'].mean():.2%}")

Sample of our customer data:
   time_on_site  pages_viewed  cart_value  previous_purchases  purchased
0     12.483571             4   28.121379                   1          1
1      9.308678             7  103.975557                   2          1
2     13.238443             7   17.937869                   1          1
3     17.615149             2    4.966511                   1          0
4      8.829233             5   71.020439                   1          1

Purchase rate: 52.20%


In [2]:
# ✅ What This Code Does:
# - Trains a logistic regression model to predict whether a customer will purchase.
# - Evaluates model accuracy on unseen test data.
# - Extracts and interprets model coefficients to explain how each feature influences purchase likelihood.


# Prepare the data
X = data[['time_on_site', 'pages_viewed', 'cart_value', 'previous_purchases']]
y = data['purchased']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create and train the logistic regression model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probability of purchase

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2%}")

# Show the coefficients (business insights)
feature_names = ['Time on Site', 'Pages Viewed', 'Cart Value', 'Previous Purchases']
coefficients = model.coef_[0]

print("\n📊 Business Insights - How each factor affects purchase probability:")
for name, coeff in zip(feature_names, coefficients):
    direction = "increases" if coeff > 0 else "decreases"
    print(f"• {name}: {coeff:.3f} - {direction} purchase probability")

print(f"\nBaseline (intercept): {model.intercept_[0]:.3f}")

Model Accuracy: 67.00%

📊 Business Insights - How each factor affects purchase probability:
• Time on Site: 0.101 - increases purchase probability
• Pages Viewed: 0.225 - increases purchase probability
• Cart Value: 0.014 - increases purchase probability
• Previous Purchases: 0.382 - increases purchase probability

Baseline (intercept): -3.247


In [3]:
# What This Code Does:
# - Predicts purchase likelihood for 4 new customers using your trained logistic regression model.
# - Prints each customer's behavior and their predicted probability of buying.
# - Categorizes customers into high, medium, and low probability groups.
# - Suggests tailored marketing actions based on their likelihood to purchase.

# Let's predict for some new customers
new_customers = pd.DataFrame({
    'time_on_site': [15, 5, 20, 8],
    'pages_viewed': [8, 2, 12, 4],
    'cart_value': [75, 25, 150, 40],
    'previous_purchases': [2, 0, 3, 1]
})

# Get predictions
purchase_probabilities = model.predict_proba(new_customers)[:, 1]
purchase_predictions = model.predict(new_customers)

print("🎯 New Customer Predictions:")
print("=" * 50)
for i, (idx, customer) in enumerate(new_customers.iterrows()):
    prob = purchase_probabilities[i]
    pred = "WILL BUY" if purchase_predictions[i] == 1 else "WON'T BUY"
    
    print(f"\nCustomer {i+1}:")
    print(f"  Time on site: {customer['time_on_site']:.1f} min")
    print(f"  Pages viewed: {customer['pages_viewed']}")
    print(f"  Cart value: ${customer['cart_value']:.0f}")
    print(f"  Previous purchases: {customer['previous_purchases']}")
    print(f"  📈 Purchase probability: {prob:.1%}")
    print(f"  🎯 Prediction: {pred}")

# Business recommendations
print("\n💡 Business Recommendations:")
high_prob_customers = np.where(purchase_probabilities > 0.7)[0]
medium_prob_customers = np.where((purchase_probabilities >= 0.3) & (purchase_probabilities <= 0.7))[0]
low_prob_customers = np.where(purchase_probabilities < 0.3)[0]

if len(high_prob_customers) > 0:
    print(f"• {len(high_prob_customers)} high-probability customers: Send premium offers")
if len(medium_prob_customers) > 0:
    print(f"• {len(medium_prob_customers)} medium-probability customers: Send discount offers")
if len(low_prob_customers) > 0:
    print(f"• {len(low_prob_customers)} low-probability customers: Focus on engagement")

🎯 New Customer Predictions:

Customer 1:
  Time on site: 15.0 min
  Pages viewed: 8
  Cart value: $75
  Previous purchases: 2
  📈 Purchase probability: 86.8%
  🎯 Prediction: WILL BUY

Customer 2:
  Time on site: 5.0 min
  Pages viewed: 2
  Cart value: $25
  Previous purchases: 0
  📈 Purchase probability: 12.6%
  🎯 Prediction: WON'T BUY

Customer 3:
  Time on site: 20.0 min
  Pages viewed: 12
  Cart value: $150
  Previous purchases: 3
  📈 Purchase probability: 99.1%
  🎯 Prediction: WILL BUY

Customer 4:
  Time on site: 8.0 min
  Pages viewed: 4
  Cart value: $40
  Previous purchases: 1
  📈 Purchase probability: 35.6%
  🎯 Prediction: WON'T BUY

💡 Business Recommendations:
• 2 high-probability customers: Send premium offers
• 1 medium-probability customers: Send discount offers
• 1 low-probability customers: Focus on engagement
